# Smart City Traffic Flow Simulator

Simulate **30 days** of hourly vehicle counts for **10 junctions**. The model uses a Poisson distribution and a rush-hour curve to represent morning and evening traffic peaks.

## Project objectives
1. Set up reproducible random data and an hourly rush-hour curve.
2. Simulate traffic with an array shaped `(days, junctions, hours)`.
3. Aggregate traffic by axis and identify peak hours with `argmax`.
4. Visualize congestion with a heatmap.
5. Compare loop-based and vectorized Poisson simulation.

**Array dimensions:** 30 days × 10 junctions × 24 hours = 7,200 junction-hour records.


## 1. Imports, configuration, and random seed

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from timeit import timeit

# Reproducible results
SEED = 42
rng = np.random.default_rng(SEED)

DAYS = 30
JUNCTIONS = 10
HOURS = 24

hours = np.arange(HOURS)
junction_names = [f"Junction {i+1}" for i in range(JUNCTIONS)]

print(f"Simulation dimensions: {DAYS} days × {JUNCTIONS} junctions × {HOURS} hours")
print(f"Total records: {DAYS * JUNCTIONS * HOURS:,}")


## 2. Create the rush-hour curve

Traffic demand is higher during the morning (around 8 AM) and evening (around 6 PM). Two smooth peaks are combined with a small baseline so that traffic does not drop to zero overnight.


In [ ]:
morning_peak = np.exp(-0.5 * ((hours - 8) / 2.0) ** 2)
evening_peak = np.exp(-0.5 * ((hours - 18) / 2.5) ** 2)

# Relative hourly demand: baseline + morning and evening rush peaks
rush_curve = 0.25 + 0.85 * morning_peak + 1.00 * evening_peak

plt.figure(figsize=(10, 4))
plt.plot(hours, rush_curve, marker="o")
plt.xticks(np.arange(0, 24, 2))
plt.xlabel("Hour of day")
plt.ylabel("Relative traffic demand")
plt.title("Hourly Rush-Hour Demand Curve")
plt.grid(alpha=0.3)
plt.show()


## 3. Simulate 30 days of traffic

Each junction has a different baseline traffic level. The expected vehicle count (`lam`) combines that baseline with the hourly rush-hour curve. A modest weekend reduction is included. `np.random.poisson` generates the observed counts.

The resulting `traffic` array has shape `(30, 10, 24)`.


In [ ]:
# Different baseline traffic levels for the 10 junctions
junction_baseline = np.linspace(35, 100, JUNCTIONS)

# A daily multiplier introduces realistic day-to-day variation
daily_factor = rng.normal(loc=1.0, scale=0.08, size=(DAYS, 1, 1))
daily_factor = np.clip(daily_factor, 0.75, 1.25)

# Lower expected traffic on weekend days (day 5 and day 6 of each week)
day_numbers = np.arange(DAYS)
weekend_factor = np.where(day_numbers % 7 >= 5, 0.80, 1.0).reshape(DAYS, 1, 1)

# Expected count for every day, junction, and hour
lam = (
    junction_baseline[None, :, None]
    * rush_curve[None, None, :]
    * daily_factor
    * weekend_factor
)

traffic = rng.poisson(lam=lam)

print("Traffic array shape:", traffic.shape)
print("Minimum count:", traffic.min())
print("Maximum count:", traffic.max())
print("Average hourly count:", round(traffic.mean(), 2))


## 4. Aggregate traffic using NumPy axes

For an array shaped `(days, junctions, hours)`:
- `axis=0` removes the day dimension.
- `axis=1` removes the junction dimension.
- `axis=2` removes the hour dimension.

The following cells calculate totals and averages for each junction and hour.


In [ ]:
# Total vehicles across all days and hours for each junction
junction_totals = traffic.sum(axis=(0, 2))

# Total vehicles across all days and junctions for each hour
hourly_totals = traffic.sum(axis=(0, 1))

# Average hourly count for each junction (averaged over days and hours)
junction_hourly_average = traffic.mean(axis=(0, 2))

print("Total vehicles by junction:")
for name, total in zip(junction_names, junction_totals):
    print(f"{name}: {total:,}")

print("\nTotal vehicles by hour (all days and junctions):")
print(hourly_totals)


## 5. Find peak hours and bottlenecks with `argmax`

In [ ]:
# Busiest hour across the complete city-wide simulation
city_peak_hour = int(np.argmax(hourly_totals))

# Busiest hour separately for each junction
junction_hour_totals = traffic.sum(axis=0)  # shape: (junctions, hours)
junction_peak_hours = np.argmax(junction_hour_totals, axis=1)

# Junction with the highest total traffic
busiest_junction = int(np.argmax(junction_totals))

print(f"City-wide peak hour: {city_peak_hour:02d}:00")
print(f"Busiest junction: {junction_names[busiest_junction]}")
print("\nPeak hour for each junction:")
for name, peak in zip(junction_names, junction_peak_hours):
    print(f"{name}: {peak:02d}:00")

# Rank junctions from highest to lowest total traffic
ranking = np.argsort(junction_totals)[::-1]
print("\nTop 3 busiest junctions:")
for index in ranking[:3]:
    print(f"{junction_names[index]}: {junction_totals[index]:,} vehicles")


## 6. Visualize traffic with a heatmap

In [ ]:
# Mean traffic per hour for each junction, averaged over 30 days
heatmap_data = traffic.mean(axis=0)  # shape: (junctions, hours)

plt.figure(figsize=(13, 6))
image = plt.imshow(heatmap_data, aspect="auto", cmap="YlOrRd", origin="upper")
plt.colorbar(image, label="Average vehicles per hour")
plt.xticks(np.arange(HOURS), [f"{h:02d}" for h in hours])
plt.yticks(np.arange(JUNCTIONS), junction_names)
plt.xlabel("Hour of day")
plt.ylabel("Junction")
plt.title("Smart City Traffic Heatmap (30-Day Average)")
plt.tight_layout()
plt.show()


## 7. Compare loop-based and vectorized simulation

Both approaches generate Poisson counts from the same expected-count array (`lam`). The loop version visits each individual day, junction, and hour. The vectorized version passes the full array to NumPy at once.

The benchmark uses fresh random generators so each timed run is independent. Run the cell more than once if you want to compare timings on your own machine.


In [ ]:
def simulate_with_loops(expected_counts, seed=123):
    local_rng = np.random.default_rng(seed)
    result = np.empty(expected_counts.shape, dtype=int)

    for day in range(expected_counts.shape[0]):
        for junction in range(expected_counts.shape[1]):
            for hour in range(expected_counts.shape[2]):
                result[day, junction, hour] = local_rng.poisson(
                    expected_counts[day, junction, hour]
                )
    return result


def simulate_vectorized(expected_counts, seed=123):
    local_rng = np.random.default_rng(seed)
    return local_rng.poisson(lam=expected_counts)


# Check that both approaches return the expected dimensions
loop_result = simulate_with_loops(lam)
vector_result = simulate_vectorized(lam)

print("Loop result shape:", loop_result.shape)
print("Vectorized result shape:", vector_result.shape)


In [ ]:
# Time each approach (lower time is faster)
NUMBER_OF_RUNS = 5

loop_seconds = timeit(
    lambda: simulate_with_loops(lam),
    number=NUMBER_OF_RUNS
) / NUMBER_OF_RUNS

vector_seconds = timeit(
    lambda: simulate_vectorized(lam),
    number=NUMBER_OF_RUNS
) / NUMBER_OF_RUNS

speedup = loop_seconds / vector_seconds if vector_seconds > 0 else float("inf")

print(f"Loop-based average: {loop_seconds:.6f} seconds")
print(f"Vectorized average: {vector_seconds:.6f} seconds")
print(f"Vectorized speedup: {speedup:.2f}x")


## 8. Project summary

This project:
- Simulated **7,200** hourly junction records across 30 days and 10 junctions.
- Used a Poisson model with morning and evening rush-hour demand.
- Used NumPy aggregation (`sum`, `mean`) and `argmax` to identify peak traffic periods and busy junctions.
- Displayed average traffic patterns using a heatmap.
- Benchmarked loop-based and vectorized simulation.

### Possible future improvements
- Use real traffic data instead of simulated counts.
- Add different traffic patterns for weekdays and holidays.
- Create an interactive dashboard with filters for date, junction, and hour.
